# ZILN Neural Net: Forward-Revenue Challenger

Trains the **Zero-Inflated LogNormal (ZILN)** neural net defined in `kkbox.ziln` as a challenger to `03a_CatBoost_and_Cox_Models.ipynb`'s Tweedie forward-revenue regressor, on the same 27-feature set. ZILN jointly predicts `(p_logit, mu, log_sigma)` from a single network — trained with a combined BCE + LogNormal-NLL loss (Wang et al., Google, 2019) — rather than the hurdle-style `P(pay) x E[amount|pay]` combination tested earlier and rejected (compounding independently-trained-model errors ate the gain there; ZILN is jointly optimized instead).

Uses the Optuna-tuned architecture from `03b_Hyperparameter_Tuning.ipynb` (`OPTUNA_BEST_PARAMS["ziln"]`), trained as a **5-seed ensemble with averaged predictions** — a standard ZILN robustness technique, since a single model's `sigma` estimate (and therefore its `exp(mu + sigma^2/2)` point estimate) can be noisy per-seed. `p_pay_feature` (the auxiliary cross-fit classifier used by the CatBoost Tweedie regressor) is deliberately **not** used here — ZILN's own `p_logit` head already jointly models payment probability, so feeding in an externally-computed P(pay) would be redundant.

This notebook is the final step before deciding forward-revenue production: it writes `results/fwd_rev_model_choice.json`, comparing test RMSE against `03a`'s `results/catboost_results.json`. Whichever model wins is what `04_Calibration_and_Business_Layer.ipynb` and `05_Final_Evaluation_Summary.ipynb` load (via `kkbox.fwd_rev.load_fwd_rev_predictor`), so **this notebook must run after `03a` and before `04`/`05`**.

In [1]:
import json
import os

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import mean_absolute_error, r2_score

from kkbox.data import columns_from_manifest
from kkbox.ziln import train_one_model, predict_ensemble

PROCESSED_DIR = os.path.join(os.getcwd(), "data", "processed")
MODELS_DIR = os.path.join(os.getcwd(), "models")
RESULTS_DIR = os.path.join(os.getcwd(), "results")
os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

train_df = pd.read_parquet(os.path.join(PROCESSED_DIR, "model_dataset_train.parquet"))
val_df = pd.read_parquet(os.path.join(PROCESSED_DIR, "model_dataset_val.parquet"))
test_df = pd.read_parquet(os.path.join(PROCESSED_DIR, "model_dataset_test.parquet"))

with open(os.path.join(PROCESSED_DIR, "feature_manifest.json")) as f:
    manifest = json.load(f)
CAT_COLS, NUM_COLS, CARDINALITIES, EMBED_DIMS = columns_from_manifest(manifest)

with open(os.path.join(RESULTS_DIR, "optuna_best_params.json")) as f:
    OPTUNA_BEST_PARAMS = json.load(f)
ziln_params = OPTUNA_BEST_PARAMS["ziln"]

ENSEMBLE_SEEDS = [42, 43, 44, 45, 46]
MAX_EPOCHS = 40    # matches the standalone validation run that confirmed stability
PATIENCE = 6

torch.set_num_threads(4)
print(f"features: {len(CAT_COLS)} categorical + {len(NUM_COLS)} numerical = {len(CAT_COLS) + len(NUM_COLS)}")
print(f"tuned hyperparameters: {ziln_params}")

features: 4 categorical + 23 numerical = 27
tuned hyperparameters: {'hidden_dim1': 256, 'hidden_dim2': 64, 'dropout1': 0.21610737493812396, 'dropout2': 0.1321766045027286, 'learning_rate': 0.0009600461498177444, 'weight_decay': 0.0008792815436511888, 'batch_size': 2048}


## Train the 5-seed ensemble

Each seed trains an independent `ZILNNet` to convergence (early stopping on validation ZILN loss) with the same tuned architecture/training hyperparameters, differing only in weight initialization and minibatch order. Checkpoints save `{state_dict, config}` together so the architecture can be reconstructed at load time without hardcoding the tuned hyperparameters anywhere else — see `kkbox.fwd_rev.load_fwd_rev_predictor`.

In [2]:
import time

architecture_config = {
    "hidden_dim1": ziln_params["hidden_dim1"],
    "hidden_dim2": ziln_params["hidden_dim2"],
    "dropout1": ziln_params["dropout1"],
    "dropout2": ziln_params["dropout2"],
}

models = []
seed_paths = []
t0 = time.time()
for seed in ENSEMBLE_SEEDS:
    model, best_val_loss, val_rmse = train_one_model(
        train_df, val_df, CAT_COLS, NUM_COLS, CARDINALITIES, EMBED_DIMS,
        hidden_dim1=ziln_params["hidden_dim1"], hidden_dim2=ziln_params["hidden_dim2"],
        dropout1=ziln_params["dropout1"], dropout2=ziln_params["dropout2"],
        learning_rate=ziln_params["learning_rate"], weight_decay=ziln_params["weight_decay"],
        batch_size=ziln_params["batch_size"], max_epochs=MAX_EPOCHS, patience=PATIENCE,
        seed=seed, verbose=True,
    )
    seed_path = f"ziln_seed{seed}.pt"
    torch.save({"state_dict": model.state_dict(), "config": architecture_config},
               os.path.join(MODELS_DIR, seed_path))
    seed_paths.append(seed_path)
    models.append(model)
    print(f"seed {seed}: best_val_ziln_loss={best_val_loss:.4f}  val_rmse={val_rmse:.2f}  "
          f"elapsed={time.time() - t0:.0f}s\n")

ensemble_manifest = {"seeds": ENSEMBLE_SEEDS, "seed_paths": seed_paths, "architecture_config": architecture_config}
with open(os.path.join(MODELS_DIR, "ziln_ensemble_manifest.json"), "w") as f:
    json.dump(ensemble_manifest, f, indent=2)

print(f"\ntrained {len(models)} models in {time.time() - t0:.0f}s total")

  [seed 42] epoch   0  val_ziln_loss=3.5834 *


  [seed 42] epoch   1  val_ziln_loss=3.4833 *


  [seed 42] epoch   2  val_ziln_loss=3.4537 *


  [seed 42] epoch   3  val_ziln_loss=3.4497 *


  [seed 42] epoch   4  val_ziln_loss=3.4347 *


...

  [seed 46] epoch  33  val_ziln_loss=3.3077 *


  [seed 46] epoch  34  val_ziln_loss=3.3072 *


  [seed 46] epoch  35  val_ziln_loss=3.3238


  [seed 46] epoch  36  val_ziln_loss=3.3044 *


seed 46: best_val_ziln_loss=3.3044  val_rmse=133.19  elapsed=1893s


trained 5 models in 1893s total


## Evaluate the ensemble (val/test)

Averages `ziln_predict()` across all 5 seeds per row (`kkbox.ziln.predict_ensemble`), then computes RMSE/MAE/R² on raw TWD — same metrics and keys as `03a`'s `catboost_results.json`, for a direct comparison.

In [3]:
def evaluate_ziln_ensemble(models, df):
    pred_raw = predict_ensemble(models, df, CAT_COLS, NUM_COLS)
    true_raw = df["fwd_rev_59d"].values
    n_bad = (~np.isfinite(pred_raw)).sum()
    assert n_bad == 0, f"{n_bad} non-finite predictions - numerical instability regression, do not trust these results"
    return {
        "fwd_rev_rmse_raw_twd": float(np.sqrt(np.mean((pred_raw - true_raw) ** 2))),
        "fwd_rev_mae_raw_twd": float(mean_absolute_error(true_raw, pred_raw)),
        "fwd_rev_r2_raw": float(r2_score(true_raw, pred_raw)),
    }


ziln_results = {
    "val": evaluate_ziln_ensemble(models, val_df),
    "test": evaluate_ziln_ensemble(models, test_df),
}
with open(os.path.join(RESULTS_DIR, "ziln_results.json"), "w") as f:
    json.dump(ziln_results, f, indent=2)

print("ZILN ensemble -- val/test (RMSE/MAE in raw TWD):")
display(pd.DataFrame({"val": ziln_results["val"], "test": ziln_results["test"]}))

ZILN ensemble -- val/test (RMSE/MAE in raw TWD):


,val,test
fwd_rev_rmse_raw_twd,133.658365,131.524435
fwd_rev_mae_raw_twd,54.872335,54.424808
fwd_rev_r2_raw,0.341990,0.348986


## Final decision: CatBoost Tweedie vs. ZILN

Compares test RMSE against `03a`'s `catboost_results.json` (must have been run already — this notebook depends on it, not the other way around) and writes `results/fwd_rev_model_choice.json`, which `04`/`05` read via `kkbox.fwd_rev.load_fwd_rev_predictor` to decide which model to load. Lower test RMSE wins; ties go to CatBoost (simpler, already in production, no reason to switch on a coin flip).

In [ ]:
catboost_results_path = os.path.join(RESULTS_DIR, "catboost_results.json")
assert os.path.exists(catboost_results_path), (
    "catboost_results.json not found - run 03a_CatBoost_and_Cox_Models.ipynb before this notebook, "
    "the final decision needs both models' test metrics."
)
with open(catboost_results_path) as f:
    catboost_results = json.load(f)

catboost_test_rmse = catboost_results["fwd_rev"]["test"]["fwd_rev_rmse_raw_twd"]
catboost_test_r2 = catboost_results["fwd_rev"]["test"]["fwd_rev_r2_raw"]
ziln_test_rmse = ziln_results["test"]["fwd_rev_rmse_raw_twd"]
ziln_test_r2 = ziln_results["test"]["fwd_rev_r2_raw"]

winner = "ziln" if ziln_test_rmse < catboost_test_rmse else "catboost"

model_choice = {
    "winner": winner,
    "decision_rule": "lower test RMSE (raw TWD) wins; ties go to catboost",
    "catboost_test_rmse": catboost_test_rmse,
    "catboost_test_r2": catboost_test_r2,
    "ziln_test_rmse": ziln_test_rmse,
    "ziln_test_r2": ziln_test_r2,
}
with open(os.path.join(RESULTS_DIR, "fwd_rev_model_choice.json"), "w") as f:
    json.dump(model_choice, f, indent=2)

print(f"CatBoost Tweedie -- test RMSE={catboost_test_rmse:.2f}  R2={catboost_test_r2:.4f}")
print(f"ZILN ensemble    -- test RMSE={ziln_test_rmse:.2f}  R2={ziln_test_r2:.4f}")
print(f"\nWINNER: {winner}")
print("(written to results/fwd_rev_model_choice.json - 04/05 will load this model for forward-revenue)")